# RSNA Knee Abnormality Detection — Image Baseline Preflight Audit

Before freezing an image-based baseline pipeline, this notebook measures several properties of the real competition DICOM data that pipeline choices depend on: whether `Fluid_Sensitive` and `Fat_Suppression` carry independent information, how many studies have usable coverage across the three anatomical planes, whether slice geometry tags are present and agree with `InstanceNumber` ordering, whether the `Laterality` tag is reliable, DICOM decode reliability, and a GPU timing probe for a frozen pretrained image encoder against the competition's runtime budget. Every result below is an aggregate count, rate, or distribution statistic — no report text, no study identifiers, and no per-study predictions.

In [ ]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from IPython.display import display

SEED = 42
IS_KAGGLE = Path("/kaggle/input").exists()
if not IS_KAGGLE:
    raise RuntimeError("This notebook runs on Kaggle only.")

DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
package_initializers = tuple(
    Path("/kaggle/input/datasets").rglob("knee_mri/__init__.py")
)
if len(package_initializers) != 1:
    raise RuntimeError("Expected exactly one attached knee_mri source package.")
sys.path.insert(0, str(package_initializers[0].parent.parent))

In [ ]:
from knee_mri.dataset import select_validated_series
from knee_mri.series_audit import (
    aggregate_group_laterality,
    audit_series,
    central_band_indices,
    fluid_fat_suppression_agreement,
    patient_lr_axis_metrics,
    plane_series_counts,
    series_transfer_syntax,
    validate_and_order_series,
)

## 1. Series Metadata Agreement and Plane Coverage (Full Corpus)

In [ ]:
train_series_df = pd.read_csv(DATA_DIR / "train_series.csv")
test_series_df = pd.read_csv(DATA_DIR / "test_series.csv")

agreement_summary = pd.DataFrame(
    [
        {"Split": "train", **fluid_fat_suppression_agreement(train_series_df)},
        {"Split": "test", **fluid_fat_suppression_agreement(test_series_df)},
    ]
).set_index("Split")

display(agreement_summary)

**Interpretation:** `agreement_rate` of 1.0 means `Fluid_Sensitive` and `Fat_Suppression` take the identical value on every series in that split — the two columns would carry no independent information, and any pipeline step that treats them as separate signals should instead treat them as one. A rate below 1.0 means the columns do sometimes disagree and remain two distinct signals worth keeping separate.

In [ ]:
def _plane_coverage(series_df: pd.DataFrame) -> pd.Series:
    plane_counts = plane_series_counts(series_df)
    present = plane_counts > 0
    coverage = {
        f"has_{plane.lower()}": float(present[plane].mean()) for plane in present.columns
    }
    coverage["has_all_three_planes"] = float(present.all(axis=1).mean())
    coverage["study_count"] = float(len(plane_counts))
    return pd.Series(coverage)

plane_coverage_summary = pd.DataFrame(
    {"train": _plane_coverage(train_series_df), "test": _plane_coverage(test_series_df)}
)

display(plane_coverage_summary)

**Interpretation:** `has_<plane>` is the fraction of studies with at least one series in that plane; `has_all_three_planes` is the fraction with usable coverage across Sagittal, Coronal, and Axial simultaneously. A high `has_all_three_planes` rate is what would make a multi-plane image baseline feasible without a large fallback-handling burden; a low rate favors a single-series baseline or an explicit per-study presence mask.

## 2. Deterministic DICOM Geometry and Decode Audit (Sampled Studies)

Auditing every series in the full corpus would decode a large fraction of the competition's 569.76 GB of DICOM data. Instead, this section audits a fixed, seeded sample of 150 studies (every series belonging to them) — a descriptive sample of that size, not a statistically validated one (no confidence intervals are computed here), chosen to stay well inside the competition's runtime budget.

In [ ]:
import importlib.util

CODEC_PACKAGES = ("pylibjpeg", "libjpeg", "openjpeg", "gdcm")
codec_availability = pd.Series(
    {package: importlib.util.find_spec(package) is not None for package in CODEC_PACKAGES},
    name="Available",
).to_frame()

display(codec_availability)

**Interpretation:** the competition data mixes JPEG Lossless, JPEG 2000, and uncompressed transfer syntaxes; decoding the compressed ones needs `libjpeg` or `openjpeg` (both part of the `pylibjpeg` plugin family) or `gdcm` available. The decode-by-transfer-syntax breakdown later in this section shows which syntaxes were actually exercised and whether failures cluster in any of them, rather than inferring codec adequacy from package presence alone.

In [ ]:
STUDY_SAMPLE_SIZE = 150
DECODE_SAMPLE_SIZE = 5

rng = np.random.default_rng(SEED)
all_study_ids = train_series_df["StudyInstanceUID"].unique()
sampled_study_ids = rng.choice(
    all_study_ids, size=min(STUDY_SAMPLE_SIZE, len(all_study_ids)), replace=False
)

train_series_dir = DATA_DIR / "train_series"
plane_lookup = train_series_df.set_index(["StudyInstanceUID", "SeriesInstanceUID"])[
    "Anatomical_Plane"
]

audit_rows = []
decode_result_rows = []
sampled_series_dirs = []
# Ephemeral only: study_id -> [(plane, resolved_laterality_call), ...] for
# this sample. Used below to compute study-level aggregate agreement rates,
# then discarded -- no study identifier is ever added to audit_rows,
# displayed, or persisted.
study_laterality_calls: dict[str, list[tuple[str | None, str | None]]] = {}
for study_id in sampled_study_ids:
    study_dir = train_series_dir / study_id
    if not study_dir.is_dir():
        continue
    for series_dir in sorted(p for p in study_dir.iterdir() if p.is_dir()):
        result = audit_series(series_dir, decode_sample_size=DECODE_SAMPLE_SIZE)
        plane = plane_lookup.get((study_id, series_dir.name))
        orientation_metrics = None
        if result.ordering_method == "geometry":
            first_path = sorted(series_dir.glob("*.dcm"))[0]
            first_header = pydicom.dcmread(first_path, stop_before_pixels=True)
            orientation_metrics = patient_lr_axis_metrics(
                first_header.ImageOrientationPatient
            )
        conservative_laterality_call = result.laterality_resolved_call
        if (
            result.header_read_failures
            or not result.laterality_tag_consistent
            or result.laterality_cross_tag_conflict
            or result.laterality_conflict
        ):
            conservative_laterality_call = None
        audit_rows.append(
            {
                "plane": plane,
                "slice_count": result.slice_count,
                "header_read_failures": result.header_read_failures,
                "has_full_geometry_tags": result.has_full_geometry_tags,
                "order_agreement": result.order_agreement,
                "ordering_usable": result.ordering_usable,
                "ordering_method": result.ordering_method,
                "laterality_tag_present_fraction": result.laterality_tag_present_fraction,
                "has_laterality_tag": result.laterality_tag is not None,
                "laterality_tag_consistent": result.laterality_tag_consistent,
                "laterality_cross_tag_conflict": result.laterality_cross_tag_conflict,
                "laterality_from_geometry": result.laterality_from_geometry,
                "laterality_conflict": result.laterality_conflict,
                "laterality_filled_by_geometry": result.laterality_filled_by_geometry,
                "conservative_laterality_call": conservative_laterality_call,
                "lr_array_axis": (
                    orientation_metrics.array_axis if orientation_metrics else None
                ),
                "lr_signed_x": (
                    orientation_metrics.signed_x if orientation_metrics else None
                ),
                "lr_dominant_abs_x": (
                    orientation_metrics.dominant_abs_x if orientation_metrics else None
                ),
                "lr_runner_up_abs_x": (
                    orientation_metrics.runner_up_abs_x if orientation_metrics else None
                ),
                "lr_dominance_gap": (
                    orientation_metrics.dominance_gap if orientation_metrics else None
                ),
                "has_pixel_spacing": result.pixel_spacing is not None,
                "pixel_spacing_row_mm": (
                    result.pixel_spacing[0] if result.pixel_spacing else None
                ),
                "pixel_spacing_col_mm": (
                    result.pixel_spacing[1] if result.pixel_spacing else None
                ),
                "decode_attempted": result.decode_attempted,
                "decode_failures": result.decode_failures,
            }
        )
        decode_result_rows.extend(
            {"transfer_syntax": transfer_syntax, "succeeded": succeeded}
            for transfer_syntax, succeeded in result.decode_results
        )
        sampled_series_dirs.append(series_dir)
        study_laterality_calls.setdefault(study_id, []).append(
            (plane, result.laterality_resolved_call)
        )

audit_df = pd.DataFrame(audit_rows)
decode_df = pd.DataFrame(decode_result_rows)

In [ ]:
resolved_agreement = audit_df["order_agreement"].dropna()
tag_missing = ~audit_df["has_laterality_tag"]
laterality_fill_among_tag_missing = (
    float(audit_df.loc[tag_missing, "laterality_filled_by_geometry"].mean())
    if tag_missing.any()
    else float("nan")
)
resolvable_laterality = audit_df.loc[
    audit_df["laterality_from_geometry"].notna() & audit_df["has_laterality_tag"],
    "laterality_conflict",
]
tag_fraction = audit_df["laterality_tag_present_fraction"]

geometry_summary = pd.Series(
    {
        "Series audited": len(audit_df),
        "Studies sampled": len(sampled_study_ids),
        "Series with >=1 unreadable header": float((audit_df["header_read_failures"] > 0).mean()),
        "Header read failure rate (of slices)": float(
            audit_df["header_read_failures"].sum() / audit_df["slice_count"].sum()
        ),
        "Geometry tag coverage": float(audit_df["has_full_geometry_tags"].mean()),
        "Ordering validation -- usable (geometry or InstanceNumber)": float(
            audit_df["ordering_usable"].mean()
        ),
        "Ordering validation -- method geometry": float(
            (audit_df["ordering_method"] == "geometry").mean()
        ),
        "Ordering validation -- method instance_number": float(
            (audit_df["ordering_method"] == "instance_number").mean()
        ),
        "Ordering validation -- unusable (neither route)": float(
            (~audit_df["ordering_usable"]).mean()
        ),
        "Order agreement -- mean signed r": float(resolved_agreement.mean()),
        "Order agreement -- fraction monotonic (|r| > 0.99)": float(
            (resolved_agreement.abs() > 0.99).mean()
        ),
        "Order agreement -- fraction r > 0": float((resolved_agreement > 0).mean()),
        "Order agreement -- fraction r < 0": float((resolved_agreement < 0).mean()),
        "Laterality tag coverage -- complete (every slice)": float((tag_fraction == 1.0).mean()),
        "Laterality tag coverage -- partial (some slices)": float(
            ((tag_fraction > 0.0) & (tag_fraction < 1.0)).mean()
        ),
        "Laterality tag coverage -- none": float((tag_fraction == 0.0).mean()),
        "Laterality tag-call coverage (consistent resolved tag call)": float(
            audit_df["has_laterality_tag"].mean()
        ),
        "Laterality resolved-call coverage (tag or geometry)": float(
            (audit_df["has_laterality_tag"] | audit_df["laterality_filled_by_geometry"]).mean()
        ),
        "Laterality tag internally consistent": float(
            audit_df["laterality_tag_consistent"].mean()
        ),
        "Laterality cross-tag conflict rate (Laterality vs ImageLaterality)": float(
            audit_df["laterality_cross_tag_conflict"].mean()
        ),
        "Laterality filled by geometry (of tag-missing series)": laterality_fill_among_tag_missing,
        "Laterality conflict rate (resolvable)": (
            float(resolvable_laterality.mean()) if len(resolvable_laterality) else float("nan")
        ),
        "PixelSpacing tag coverage": float(audit_df["has_pixel_spacing"].mean()),
        "Decode failure rate": float(
            audit_df["decode_failures"].sum() / audit_df["decode_attempted"].sum()
        ),
    },
    name="Value",
).to_frame()

display(geometry_summary)

**Interpretation:** `Series with >=1 unreadable header` and `Header read failure rate (of slices)` above 0 mean the audit encountered a malformed or unreadable `.dcm` file. Such a file is counted rather than allowed to abort the run, and any series containing one is marked unusable for ordering — a series that cannot be read in full is not one whose slice order can be relied on. `Ordering validation -- usable` is the strict gate: a series counts as usable only if its geometry is finite, parseable, non-degenerate, mutually consistent and pairwise distinguishable, or, failing that, every slice carries a unique parseable `InstanceNumber`. Unlike `Geometry tag coverage` above, which only checks that tags are *present*, an unusable series here is never quietly ordered by filename instead. `Ordering validation -- unusable` above 0 means the same-plane retry and missing-plane fallback do real work rather than existing as unused defensive code. A high `fraction monotonic` means `InstanceNumber` order and true DICOM-geometry order agree — consistently, in one direction or its exact reverse — for nearly every series. `fraction r > 0` versus `fraction r < 0` does **not** mean "same real-world direction" versus "reversed" across series: each series' sign is relative to its own `ImageOrientationPatient`-derived normal, which is not canonicalized to a shared anatomical axis. The conclusion these numbers actually support is narrower than it first appears — `InstanceNumber` is adequate for selecting a symmetric central-band sample pooled by an order-invariant operation such as a mean, but not for any design that assumes a consistent physical direction across series without first canonicalizing geometry to a fixed axis. The three `Laterality tag coverage` rows separate series with a valid tag on every slice, some slices, or none. `Laterality cross-tag conflict rate` is a stricter and quite different signal from `Laterality tag internally consistent`: it catches a single slice carrying two disagreeing valid tags, which a same-slice precedence rule would otherwise resolve silently. `Laterality filled by geometry (of tag-missing series)` is the number that actually measures whether the geometry fallback recovers a call the tag alone could not make. `PixelSpacing tag coverage` below 1.0 means physical-millimetre cropping needs a defined fallback. `Decode failure rate` above 0 means the pipeline needs an explicit fallback for unreadable slices rather than assuming every decode succeeds.

In [ ]:
def _summarize_group_agreement(results):
    resolved_results = [r for r in results if r.resolved > 0]
    return {
        "Studies": len(results),
        "Studies with >=1 resolved series": sum(1 for r in results if r.resolved > 0),
        "Resolved-studies consistency rate": (
            sum(r.consistent for r in resolved_results) / len(resolved_results)
            if resolved_results
            else float("nan")
        ),
    }


study_level_agreement = []
plane_level_agreement = []
for entries in study_laterality_calls.values():
    all_calls = [call for _, call in entries]
    study_level_agreement.append(aggregate_group_laterality(all_calls))

    # NOT the frozen series selector (select_primary_series prefers
    # Fluid_Sensitive == 1 within a plane) -- this is just the first
    # sampled series encountered per plane, a cheap proxy for "does
    # restricting to one series per plane change the agreement rate" only.
    first_per_plane_calls = []
    seen_planes = set()
    for plane, call in entries:
        if plane is None or plane in seen_planes:
            continue
        seen_planes.add(plane)
        first_per_plane_calls.append(call)
    plane_level_agreement.append(aggregate_group_laterality(first_per_plane_calls))

study_laterality_summary = pd.DataFrame(
    {
        "All sampled series in study": _summarize_group_agreement(study_level_agreement),
        "First series per plane (not the frozen selector)": _summarize_group_agreement(
            plane_level_agreement
        ),
    }
)

display(study_laterality_summary)

**Interpretation:** aggregates the series-level laterality calls up to the study the real pipeline actually needs consistency at -- one knee per study, so every resolved series within a study should agree. "Studies with >=1 resolved series" is the fraction where at least one series (tag or geometry) resolves a call at all; "Resolved-studies consistency rate" is, among those, how often every resolved series in the study agrees. "All sampled series in study" is the number that matters for a study-wide laterality consensus derived from every available series header, regardless of which series an image selector later picks. "First series per plane" is *not* the actual frozen series selector (which prefers `Fluid_Sensitive == 1` within a plane) -- it only checks whether restricting to one arbitrary series per plane changes the agreement rate, and should not be read as "the series the compact multi-plane design would select."

## 2b. Patient Left–Right Axis Orientation

In [ ]:
ORIENTATION_THRESHOLDS = (0.80, 0.85, 0.90, 0.95)
orientation_df = audit_df.dropna(subset=["lr_dominant_abs_x"]).copy()
orientation_groups = {
    plane: orientation_df[orientation_df["plane"] == plane]
    for plane in sorted(orientation_df["plane"].dropna().unique())
}
orientation_groups["All planes"] = orientation_df


def _orientation_distribution(df: pd.DataFrame) -> pd.Series:
    dominant = df["lr_dominant_abs_x"]
    gap = df["lr_dominance_gap"]
    return pd.Series(
        {
            "Series": len(df),
            "Dominant |X| minimum": dominant.min(),
            "Dominant |X| 5th percentile": dominant.quantile(0.05),
            "Dominant |X| median": dominant.median(),
            "Dominant |X| maximum": dominant.max(),
            "Dominance gap minimum": gap.min(),
            "Dominance gap 5th percentile": gap.quantile(0.05),
            "Dominance gap median": gap.median(),
        }
    )


def _orientation_threshold_counts(df: pd.DataFrame) -> pd.Series:
    counts = {"Series": len(df)}
    for threshold in ORIENTATION_THRESHOLDS:
        counts[f"Below {threshold:.2f}"] = int(
            (df["lr_dominant_abs_x"] < threshold).sum()
        )
    return pd.Series(counts)


orientation_distribution_summary = pd.DataFrame(
    {name: _orientation_distribution(group) for name, group in orientation_groups.items()}
)
orientation_threshold_summary = pd.DataFrame(
    {name: _orientation_threshold_counts(group) for name, group in orientation_groups.items()}
)
orientation_axis_summary = pd.crosstab(
    orientation_df["plane"], orientation_df["lr_array_axis"], margins=True
)
orientation_sign_source = orientation_df.dropna(
    subset=["conservative_laterality_call", "lr_array_axis", "lr_signed_x"]
).copy()
orientation_sign_source["direction_sign"] = np.where(
    orientation_sign_source["lr_signed_x"] > 0, "positive", "negative"
)
orientation_sign_summary = pd.crosstab(
    [orientation_sign_source["plane"], orientation_sign_source["conservative_laterality_call"]],
    [orientation_sign_source["lr_array_axis"], orientation_sign_source["direction_sign"]],
)

display(orientation_distribution_summary)
display(orientation_threshold_summary)
display(orientation_axis_summary)
display(orientation_sign_summary)

**Interpretation:** the dominant patient-X component identifies whether left–right position varies most strongly along image columns, image rows, or the geometry-ordered slice stack. The magnitude and dominance-gap summaries show how clearly that axis is separated from the alternatives; the threshold table reports exact counts below each predefined candidate rather than selecting a cutoff from model performance. The axis and signed-direction tables verify whether storage conventions differ by plane or knee side. Only geometry-validated series contribute, and all displayed and persisted results are aggregate counts or distributions without study or series identifiers.

## 2c. Series Ranking, Validation, and Retry (Sampled Studies)

Measures the production selection contract directly: for each of the same 150 sampled studies and each of the three candidate planes, rank that study's series for the plane — fluid-sensitive preferred, then most slices, then `SeriesInstanceUID` as a deterministic tie-break — and try each in order against the strict ordering gate until one validates or the candidates run out. This answers the question the design actually turns on: how often does the top-ranked candidate simply work, how often is a retry genuinely needed, and how often does a plane end up with no usable candidate at all.

In [ ]:
PLANES = ("Sagittal", "Coronal", "Axial")

plane_selection_rows = []
for study_id in sampled_study_ids:
    for plane in PLANES:
        selection = select_validated_series(train_series_df, train_series_dir, study_id, plane)
        plane_selection_rows.append(
            {
                "plane": plane,
                "resolved": selection.series_instance_uid is not None,
                "ordering_method": selection.ordering_method,
                "candidates_tried": selection.candidates_tried,
                "retry_needed": selection.candidates_tried > 1,
            }
        )

plane_selection_df = pd.DataFrame(plane_selection_rows)


def _plane_selection_summary(df: pd.DataFrame) -> pd.Series:
    resolved = df["resolved"]
    resolved_rows = df.loc[resolved]
    return pd.Series(
        {
            "Study-plane pairs": float(len(df)),
            "Resolved (usable series found)": float(resolved.mean()),
            "Retry needed (of resolved)": (
                float(resolved_rows["retry_needed"].mean()) if len(resolved_rows) else float("nan")
            ),
            "Method geometry (of resolved)": (
                float((resolved_rows["ordering_method"] == "geometry").mean())
                if len(resolved_rows)
                else float("nan")
            ),
            "Method instance_number (of resolved)": (
                float((resolved_rows["ordering_method"] == "instance_number").mean())
                if len(resolved_rows)
                else float("nan")
            ),
        }
    )


plane_selection_summary = pd.DataFrame(
    {
        plane: _plane_selection_summary(plane_selection_df[plane_selection_df["plane"] == plane])
        for plane in PLANES
    }
)
plane_selection_summary["All planes"] = _plane_selection_summary(plane_selection_df)

display(plane_selection_summary)

**Interpretation:** `Resolved` below 1.0 for a plane means that fraction of studies would trigger the missing-plane fallback (excluded from the mean, presence flag 0) for that plane under the real contract -- not a hypothetical. `Retry needed (of resolved)` above 0 means the top-ranked candidate alone is not sufficient and same-plane retry is doing real work, not just existing as unused defensive code. The method split shows how often the eventual winning candidate needed the `InstanceNumber` fallback route rather than geometry.

In [ ]:
pixel_spacing_summary = audit_df[
    ["pixel_spacing_row_mm", "pixel_spacing_col_mm"]
].describe().T[["mean", "std", "min", "max"]]
slice_count_summary = (
    audit_df["slice_count"]
    .describe()[["mean", "std", "min", "50%", "max"]]
    .rename({"50%": "median"})
    .to_frame(name="Value")
)

display(pixel_spacing_summary)
display(slice_count_summary)

**Interpretation:** a wide `pixel_spacing` range confirms physical-extent (millimeter-based) cropping is necessary rather than a fixed-pixel crop, since a fixed pixel window would cover a different real-world area per study. The `slice_count` distribution informs how large a central-band slice sample can be without exceeding what most series actually contain.

In [ ]:
decode_by_transfer_syntax = decode_df.groupby("transfer_syntax")["succeeded"].agg(
    attempted="count", failures=lambda outcomes: int((~outcomes).sum())
)
decode_by_transfer_syntax["failure_rate"] = (
    decode_by_transfer_syntax["failures"] / decode_by_transfer_syntax["attempted"]
)

display(decode_by_transfer_syntax)

**Interpretation:** breaks the overall `Decode failure rate` down by `TransferSyntaxUID`, so a failure pattern concentrated in one compressed syntax (versus spread evenly, or absent) is visible directly rather than inferred from which codec packages happen to be importable.

### Corpus-Wide Transfer-Syntax Census

In [ ]:
# Header-only transfer-syntax census across every series in both splits.
# The seeded 150-study sample above observed only uncompressed data, which
# cannot establish what the rest of the corpus stores -- reading one header
# per series is what makes a full-corpus census affordable, and it is a
# storage-format census only, not a decode-reliability measurement.
census_rows = []
census_unreadable_series = 0
census_empty_series = 0

for split_name, split_series_df, split_dir in (
    ("train", train_series_df, DATA_DIR / "train_series"),
    ("test", test_series_df, DATA_DIR / "test_series"),
):
    for census_study_id in split_series_df["StudyInstanceUID"].unique():
        census_study_dir = split_dir / census_study_id
        if not census_study_dir.is_dir():
            continue
        for census_series_dir in sorted(
            p for p in census_study_dir.iterdir() if p.is_dir()
        ):
            try:
                census_uid = series_transfer_syntax(census_series_dir)
            except FileNotFoundError:
                census_empty_series += 1
                continue
            if census_uid is None:
                census_unreadable_series += 1
                continue
            census_rows.append({"split": split_name, "transfer_syntax": census_uid})

census_df = pd.DataFrame(census_rows, columns=["split", "transfer_syntax"])
transfer_syntax_census = (
    census_df.groupby(["transfer_syntax", "split"]).size().unstack("split", fill_value=0)
)
for split_name in ("train", "test"):
    if split_name not in transfer_syntax_census.columns:
        transfer_syntax_census[split_name] = 0
transfer_syntax_census = transfer_syntax_census[["train", "test"]]
transfer_syntax_census["series"] = transfer_syntax_census.sum(axis=1)
transfer_syntax_census["compressed"] = [
    pydicom.uid.UID(uid).is_compressed for uid in transfer_syntax_census.index
]
transfer_syntax_census["syntax_name"] = [
    pydicom.uid.UID(uid).name for uid in transfer_syntax_census.index
]

census_coverage = pd.Series(
    {
        "Series censused": float(len(census_df)),
        "Series with no readable header": float(census_unreadable_series),
        "Series with no .dcm file": float(census_empty_series),
        "Distinct transfer syntaxes": float(census_df["transfer_syntax"].nunique()),
        "Compressed syntaxes observed": float(
            sum(
                pydicom.uid.UID(uid).is_compressed
                for uid in census_df["transfer_syntax"].unique()
            )
        ),
    },
    name="Value",
).to_frame()

display(transfer_syntax_census)
display(census_coverage)

**Interpretation:** the sampled audit above measures decode reliability on 150 studies; this censuses *storage format* across every series in both splits, which is the reproducible way to learn which transfer syntaxes the pipeline must actually be able to read. Any row with `compressed` true requires a working decoder at inference time, so a non-zero compressed count here is what makes offline codec vendoring mandatory rather than precautionary -- and a census showing only uncompressed storage across the whole corpus would retire that requirement on evidence instead of assumption. `Series with no readable header` and `Series with no .dcm file` bound how much of the corpus this census could not classify; both should be at or near zero, and a non-zero value is a data-path problem to diagnose, not a rounding detail. This section deliberately does not decode any compressed pixel data: that is a separate, later step that needs the vendored wheels in place first.

## 3. Frozen DINOv2-Small Load and GPU Timing Probe

Loads the frozen, offline-vendored `facebook/dinov2-small` model attached as a Kaggle Model source and times both DICOM decode/preprocessing and the GPU forward pass on a sample of the same sampled series, to project total wall-clock time against the competition's runtime budget before any pipeline design is frozen.

In [ ]:
import importlib.metadata

import torch
from transformers import AutoModel

if not torch.cuda.is_available():
    raise RuntimeError("Expected a GPU-enabled kernel for the DINOv2 timing probe.")


def _find_dinov2_dir(root: Path) -> Path:
    for config_path in root.rglob("config.json"):
        try:
            config = json.loads(config_path.read_text())
        except (OSError, json.JSONDecodeError):
            continue
        if config.get("model_type") == "dinov2":
            return config_path.parent
    raise RuntimeError("Could not find an attached DINOv2 model source.")


DEVICE = torch.device("cuda")
dinov2_dir = _find_dinov2_dir(Path("/kaggle/input"))
dinov2 = AutoModel.from_pretrained(str(dinov2_dir), local_files_only=True).to(DEVICE).eval()
for parameter in dinov2.parameters():
    parameter.requires_grad_(False)

cuda_major, cuda_minor = torch.cuda.get_device_capability(0)
GPU_COMPATIBLE = f"sm_{cuda_major}{cuda_minor}" in torch.cuda.get_arch_list()

environment_summary = pd.Series(
    {
        "torch version": importlib.metadata.version("torch"),
        "transformers version": importlib.metadata.version("transformers"),
        "CUDA device": torch.cuda.get_device_name(0),
        "CUDA compute capability": f"{cuda_major}.{cuda_minor}",
        "GPU compatible with installed PyTorch build": GPU_COMPATIBLE,
        "DINOv2 parameters": sum(p.numel() for p in dinov2.parameters()),
    },
    name="Value",
).to_frame()

display(environment_summary)

**Interpretation:** confirms the offline `model_sources` vendoring pattern works under `enable_internet: false` and records the exact runtime environment the timing numbers below were measured on, since GPU timing is only meaningful alongside the hardware/library versions it was measured with. Kaggle's shared GPU pool can allocate an older accelerator (e.g. a P100) whose compute capability the preinstalled PyTorch build no longer supports -- `GPU compatible with installed PyTorch build` makes that visible instead of the run crashing partway through.

In [ ]:
IMAGE_SIZE = 336
GPU_TIMING_SERIES_SAMPLE = 30
GPU_BATCH_SIZE = 32


def _percentile_normalize(pixels: np.ndarray) -> np.ndarray:
    low, high = np.percentile(pixels, [1, 99])
    if high <= low:
        return np.zeros_like(pixels, dtype=np.float32)
    clipped = np.clip(pixels, low, high)
    return ((clipped - low) / (high - low)).astype(np.float32)


def _to_model_input(pixels: np.ndarray) -> torch.Tensor:
    tensor = torch.from_numpy(pixels).unsqueeze(0).unsqueeze(0)
    resized = torch.nn.functional.interpolate(
        tensor, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False
    )
    return resized.repeat(1, 3, 1, 1).squeeze(0)


timing_series_dirs = sampled_series_dirs[:GPU_TIMING_SERIES_SAMPLE]

decode_seconds = None
gpu_seconds = None
batch_tensors = []
if GPU_COMPATIBLE:
    decode_start = time.perf_counter()
    for series_dir in timing_series_dirs:
        validation = validate_and_order_series(series_dir)
        # Timing only needs *some* representative slices, not a validated
        # order (decode/GPU cost doesn't depend on slice order) -- fall
        # back to filename order for this narrow purpose only, same as
        # audit_series's own decode-reliability sampling.
        ordered_paths = (
            validation.ordered_paths
            if validation.usable and validation.ordered_paths is not None
            else sorted(series_dir.glob("*.dcm"))
        )
        for index in central_band_indices(len(ordered_paths), DECODE_SAMPLE_SIZE):
            dataset = pydicom.dcmread(ordered_paths[index])
            pixels = _percentile_normalize(dataset.pixel_array.astype(np.float32))
            batch_tensors.append(_to_model_input(pixels))
    decode_seconds = time.perf_counter() - decode_start

    gpu_seconds = 0.0
    for start in range(0, len(batch_tensors), GPU_BATCH_SIZE):
        chunk = torch.stack(batch_tensors[start : start + GPU_BATCH_SIZE]).to(DEVICE)
        torch.cuda.synchronize()
        gpu_start = time.perf_counter()
        with torch.no_grad():
            _ = dinov2(pixel_values=chunk, interpolate_pos_encoding=True)
        torch.cuda.synchronize()
        gpu_seconds += time.perf_counter() - gpu_start

In [ ]:
if GPU_COMPATIBLE:
    slices_processed = len(batch_tensors)
    series_processed = len(timing_series_dirs)
    seconds_per_series = (decode_seconds + gpu_seconds) / series_processed

    # Any submittable design fits only on the 58 gold-labeled train studies
    # and infers on the documented ~1,300-study hidden test set
    # (docs/1_instructions.md) -- not all 4,407 train studies, most of which
    # carry no human label and are never encoded by this pipeline (Phase 2's
    # no-go verdict against weak-label training). "Three series per study"
    # is the compact multi-plane candidate (one selected series per
    # anatomical plane), not "all series per study", which includes
    # redundant series within a plane no candidate design would encode.
    GOLD_LABELED_STUDY_COUNT = 58
    DOCUMENTED_HIDDEN_TEST_STUDY_COUNT = 1300
    workload_study_count = GOLD_LABELED_STUDY_COUNT + DOCUMENTED_HIDDEN_TEST_STUDY_COUNT

    timing_summary = pd.Series(
        {
            "GPU timing measured": True,
            "Slices processed (timing sample)": slices_processed,
            "Series processed (timing sample)": series_processed,
            "Decode seconds per slice": decode_seconds / slices_processed,
            "GPU forward seconds per slice": gpu_seconds / slices_processed,
            "Measured seconds per series (decode + GPU forward only)": seconds_per_series,
            "Workload studies (58 gold train + ~1,300 documented hidden test)": (
                workload_study_count
            ),
            "Lower-bound hours -- one series per study": (
                seconds_per_series * workload_study_count / 3600
            ),
            "Lower-bound hours -- three series per study (compact multi-plane)": (
                seconds_per_series * 3 * workload_study_count / 3600
            ),
            "Competition runtime budget (hours)": 9.0,
        },
        name="Value",
    ).to_frame()
else:
    timing_summary = pd.Series(
        {
            "GPU timing measured": False,
            "Reason": "Allocated GPU compute capability unsupported by installed PyTorch",
        },
        name="Value",
    ).to_frame()

display(timing_summary)

**Interpretation:** the two lower-bound-hours rows compare the measured cost of DICOM decode plus the frozen encoder's GPU forward pass -- and only that -- for a single-series design versus the compact three-series (one per plane) multi-plane candidate, against the actual workload (58 gold-labeled train studies plus the documented hidden test set). This explicitly excludes series-selection logic, host-to-device transfer beyond the timed batch, model loading, embedding materialization/storage, the classifier head's training or CV, any training dataloader, and I/O contention from other concurrent work -- it is a lower bound on one component of total runtime, not an end-to-end estimate. The margin against the competition's 9-hour budget (roughly 55x for the measured component alone) is large enough to support a narrower conclusion: encoder runtime specifically is not a reason to prefer the single-series design over the compact three-series one. It does not by itself guarantee the full pipeline finishes in any particular time. If `GPU compatible with installed PyTorch build` was `False` above, `GPU timing measured` is `False` here and no projection could be computed this run.

## 4. Persisted Aggregate Summary

In [ ]:
import json as _json


def _to_jsonable(value):
    if isinstance(value, (pd.Series, pd.DataFrame)):
        return _json.loads(value.to_json())
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value


preflight_summary = {
    "fluid_fat_suppression_agreement": _to_jsonable(agreement_summary),
    "plane_coverage": _to_jsonable(plane_coverage_summary),
    "codec_availability": _to_jsonable(codec_availability),
    "geometry_and_decode_audit": _to_jsonable(geometry_summary),
    "study_laterality_agreement": _to_jsonable(study_laterality_summary),
    "orientation_distribution": _to_jsonable(orientation_distribution_summary),
    "orientation_threshold_counts": _to_jsonable(orientation_threshold_summary),
    "orientation_axis_counts": _to_jsonable(orientation_axis_summary),
    "orientation_sign_by_plane_and_side": _to_jsonable(orientation_sign_summary),
    "plane_selection": _to_jsonable(plane_selection_summary),
    "decode_by_transfer_syntax": _to_jsonable(decode_by_transfer_syntax),
    "transfer_syntax_census": _to_jsonable(transfer_syntax_census),
    "transfer_syntax_census_coverage": _to_jsonable(census_coverage),
    "pixel_spacing": _to_jsonable(pixel_spacing_summary),
    "slice_count": _to_jsonable(slice_count_summary),
    "environment": _to_jsonable(environment_summary),
    "gpu_timing": _to_jsonable(timing_summary),
}

with open("/kaggle/working/preflight_audit_summary.json", "w") as handle:
    _json.dump(preflight_summary, handle, indent=2)

**Interpretation:** every aggregate table above, gathered into one file so it can be retrieved from `/kaggle/working` after the run completes -- Kaggle does not expose a notebook-type kernel's rendered `display()` output through its file-download API, only files written to `/kaggle/working` and a plain stderr/traceback log.

## 5. Summary

This audit answers, with real measurements on this competition's data, the open questions raised before freezing an image baseline pipeline: whether `Fluid_Sensitive` and `Fat_Suppression` are redundant, how much multi-plane coverage exists per study, whether `InstanceNumber` order is a reliable substitute for geometry-based ordering, how reliable the `Laterality` tag is, DICOM decode reliability, and a measured GPU runtime projection against the competition's budget. These numbers, not assumptions carried over from public reference notebooks, are what the frozen pipeline design should be built on.